In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences


KeyboardInterrupt: 

In [2]:
df = pd.read_excel("My_datasets/english_to_urdu_translation_dataset.xlsx")


In [5]:
df.head(1)

,eng,urdu
0,the book of the generation of jesus christ th...,یسوع مسیح ابن داود ابن ابرہام کا نسب نامہ


In [7]:
def preprocess_sentence(w):
    w = str(w).lower().strip()
    w='<start> ' + w + ' <end>'
    return w

df['eng'] = df['eng'].apply(preprocess_sentence)
df['urdu'] = df['urdu'].apply(preprocess_sentence)

In [10]:
def tokenaize(lang):
    tokenizer = Tokenizer(filters='')
    tokenizer.fit_on_texts(lang)
    tensor=tokenizer.texts_to_sequences(lang)
    tensor = pad_sequences(tensor,padding='post')
    return tensor,tokenizer
input_tensor,inp_lang = tokenaize(df['eng'])
target_tensor,targ_lang = tokenaize(df['urdu'])

max_length_inp = input_tensor.shape[1]
max_length_targ = target_tensor.shape[1]

In [11]:
class Encoder(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim, enc_units, batch_sz):
        super(Encoder, self).__init__()
        self.batch_sz = batch_sz
        self.enc_units = enc_units
        self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)
        self.gru = tf.keras.layers.GRU(self.enc_units, return_sequences=True, 
                                       return_state=True, recurrent_initializer='glorot_uniform')

    def call(self, x, hidden):
        x = self.embedding(x)
        output, state = self.gru(x, initial_state=hidden)
        return output, state

    def initialize_hidden_state(self):
        return tf.zeros((self.batch_sz, self.enc_units))

In [12]:
class Decoder(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim, dec_units, batch_sz):
        super(Decoder, self).__init__()
        self.batch_sz = batch_sz
        self.dec_units = dec_units
        self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)
        self.gru = tf.keras.layers.GRU(self.dec_units, return_sequences=True, 
                                       return_state=True, recurrent_initializer='glorot_uniform')
        self.fc = tf.keras.layers.Dense(vocab_size)

        # Attention layer
        self.attention = tf.keras.layers.AdditiveAttention()

    def call(self, x, hidden, enc_output):
        # enc_output shape == (batch_size, max_length, hidden_size)
        query = tf.expand_dims(hidden, 1)
        context_vector = self.attention([query, enc_output])

        x = self.embedding(x)
        x = tf.concat([context_vector, x], axis=-1)

        output, state = self.gru(x)
        output = tf.reshape(output, (-1, output.shape[2]))
        x = self.fc(output)

        return x, state

In [13]:
optimizer = tf.keras.optimizers.Adam()
loss_object = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True, reduction='none')

def loss_function(real, pred):
    mask = tf.math.logical_not(tf.math.equal(real, 0))
    loss_ = loss_object(real, pred)
    mask = tf.cast(mask, dtype=loss_.dtype)
    loss_ *= mask
    return tf.reduce_mean(loss_)

@tf.function
def train_step(inp, targ, enc_hidden):
    loss = 0
    with tf.GradientTape() as tape:
        enc_output, enc_hidden = encoder(inp, enc_hidden)
        dec_hidden = enc_hidden
        dec_input = tf.expand_dims([targ_lang.word_index['<start>']] * BATCH_SIZE, 1)

        for t in range(1, targ.shape[1]):
            predictions, dec_hidden = decoder(dec_input, dec_hidden, enc_output)
            loss += loss_function(targ[:, t], predictions)
            dec_input = tf.expand_dims(targ[:, t], 1)

    batch_loss = (loss / int(targ.shape[1]))
    variables = encoder.trainable_variables + decoder.trainable_variables
    gradients = tape.gradient(loss, variables)
    optimizer.apply_gradients(zip(gradients, variables))
    return batch_loss

In [14]:
def translate(sentence):
    sentence = preprocess_sentence(sentence)
    inputs = [inp_lang.word_index[i] for i in sentence.split(' ')]
    inputs = pad_sequences([inputs], maxlen=max_length_inp, padding='post')
    inputs = tf.convert_to_tensor(inputs)

    result = ''
    hidden = [tf.zeros((1, units))]
    enc_out, enc_hidden = encoder(inputs, hidden)
    dec_hidden = enc_hidden
    dec_input = tf.expand_dims([targ_lang.word_index['<start>']], 0)

    for t in range(max_length_targ):
        predictions, dec_hidden = decoder(dec_input, dec_hidden, enc_out)
        predicted_id = tf.argmax(predictions[0]).numpy()
        result += targ_lang.index_word[predicted_id] + ' '
        if targ_lang.index_word[predicted_id] == '<end>':
            return result, sentence
        dec_input = tf.expand_dims([predicted_id], 0)

    return result, sentence

In [16]:
# Define these parameters first
BATCH_SIZE = 64
embedding_dim = 256
units = 1024  # This is the 'units' the error was looking for
vocab_inp_size = len(inp_lang.word_index) + 1
vocab_tar_size = len(targ_lang.word_index) + 1

# Initialize the models
encoder = Encoder(vocab_inp_size, embedding_dim, units, BATCH_SIZE)
decoder = Decoder(vocab_tar_size, embedding_dim, units, BATCH_SIZE)

In [17]:
result, sentence = translate('how are you')

In [18]:
result

'مطمئن ٹام باز بستر گنوا رو اکلوتے قابو شادمان برداشت رندوں برداشت رندوں برداشت رندوں برداشت رندوں برداشت رندوں برداشت رندوں برداشت رندوں برداشت رندوں برداشت رندوں برداشت رندوں برداشت رندوں برداشت رندوں برداشت رندوں برداشت رندوں برداشت رندوں برداشت رندوں برداشت رندوں برداشت رندوں برداشت رندوں برداشت رندوں برداشت رندوں برداشت رندوں برداشت رندوں برداشت رندوں برداشت رندوں برداشت رندوں برداشت رندوں برداشت رندوں برداشت رندوں برداشت رندوں برداشت رندوں برداشت رندوں برداشت رندوں برداشت رندوں برداشت رندوں برداشت رندوں برداشت رندوں برداشت رندوں برداشت '

In [21]:
%pip install -q -U datasets transformers sentencepiece accelerate

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import datasets
print(datasets.__file__)

e:\Deep Learning\tf_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


e:\Deep Learning\tf_env\Lib\site-packages\datasets\__init__.py


In [8]:
import pandas as pd
from datasets import Dataset
from transformers import (
    T5Tokenizer,
    T5ForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq
)


# Clean column names
df.columns = df.columns.str.strip().str.lower()

print("Columns found:", df.columns)

# Rename correctly (adjust if needed)
df = df.rename(columns={
    "eng": "src",
    "english": "src",
    "urdu": "tgt"
})

# Drop missing values
df = df.dropna()

# Ensure string type
df["src"] = df["src"].astype(str)
df["tgt"] = df["tgt"].astype(str)

print(df.head())

# Convert to HF dataset
dataset = Dataset.from_pandas(df)

# Train-test split
dataset = dataset.train_test_split(test_size=0.1)

# =========================
# 2. LOAD MODEL + TOKENIZER
# =========================
model_name = "t5-small"

tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

# =========================
# 3. PREPROCESSING
# =========================
max_input_length = 128
max_target_length = 128

def preprocess(example):
    inputs = ["translate English to Urdu: " + x for x in example["src"]]

    model_inputs = tokenizer(
        inputs,
        max_length=max_input_length,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        example["tgt"],
        max_length=max_target_length,
        truncation=True,
        padding="max_length"
    )

    # IMPORTANT FIX (loss stability)
    label_ids = []
    for label in labels["input_ids"]:
        label_ids.append([
            token if token != tokenizer.pad_token_id else -100
            for token in label
        ])

    model_inputs["labels"] = label_ids
    return model_inputs

tokenized_dataset = dataset.map(preprocess, batched=True)

# =========================
# 4. DATA COLLATOR
# =========================
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

# =========================
# 5. TRAINING ARGS (SAFE VERSION)
# =========================
training_args = Seq2SeqTrainingArguments(
    output_dir="./results",
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    report_to=[]
)

# =========================
# 6. TRAINER (FIXED FOR YOUR VERSION)
# =========================
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    data_collator=data_collator
)

# =========================
# 7. TRAIN
# =========================
trainer.train()

# =========================
# 8. SAVE MODEL
# =========================
model.save_pretrained("eng-urdu-model")
tokenizer.save_pretrained("eng-urdu-model")

print("Training complete & model saved!")

Columns found: Index(['src', 'tgt'], dtype='object')
                                                 src  \
0  the book of the generation of jesus christ  th...   
1  abraham begat isaac  and isaac begat jacob  an...   
2  and judas begat phares and zara of thamar  and...   
3  and aram begat aminadab  and aminadab begat na...   
4  and salmon begat booz of rachab  and booz bega...   

                                                 tgt  
0          یسوع مسیح ابن داود ابن ابرہام کا نسب نامہ  
1  ابراہام سے اضحاق پیدا ہوا اور اضحاق سے یعقوب پ...  
2  اور یہوداہ سے فارص اور زارح تمر سے پیدا ہوئے ا...  
3  اور رام سے عمینداب پیدا ہوا اور عمینداب سے نحس...  
4  اور سلمون سے بوعز راحب سے پیدا ہوا اور بوعز سے...  


Map: 100%|██████████| 911/911 [00:00<00:00, 5979.98 examples/s]
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
e:\Deep Learning\tf_env\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


KeyboardInterrupt: 

In [4]:
import transformers
print(transformers.__version__)

5.5.4


In [ ]:
from transformers import T5Tokenizer, T5ForConditionalGeneration

model = T5ForConditionalGeneration.from_pretrained("eng-urdu-model")
tokenizer = T5Tokenizer.from_pretrained("eng-urdu-model")

text = "translate English to Urdu: How are you?"

inputs = tokenizer(text, return_tensors="pt")

outputs = model.generate(**inputs, max_length=50)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))